In [ ]:
%matplotlib qt

import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np

from espm.estimators import SmoothNMF

In [ ]:
bcf_path = "X3-13MAY22_MAP06.bcf"
signals = hs.load(bcf_path)
eds_sig = signals[1].isig[0.1:]

s = eds_sig.rebin(scale=(8, 8, 1))
s.set_signal_type("EDS_espm")

In [ ]:
s.set_analysis_parameters(
    thickness=10e-5,
    density=4.1,
    detector_type="SDD_efficiency.txt",
    width_slope=0.01,
    width_intercept=0.065,
    geom_eff=None,
    xray_db="200keV_xrays.json",
)

s.change_dtype("float64")

original_mean = s.data.mean((0, 1))

In [ ]:
def run_decomp(name, build_g_func):
    print(f"\n--- Running decomposition for: {name} ---")
    s_run = s.deepcopy()
    build_g_func(s_run)

    estimator = SmoothNMF(
        n_components=3,
        G=s_run.G,
        max_iter=500,
        tol=1e-5,
        init="nndsvdar",
        random_state=42,
        hspy_comp=True,
    )

    s_run.decomposition(algorithm=estimator)

    G_est = estimator.G_
    W_est = estimator.W_
    H_est = estimator.H_

    reconstructed = G_est @ W_est @ H_est
    reconstructed_mean = reconstructed.mean(1)

    mse = np.mean((reconstructed_mean - original_mean) ** 2)
    mae = np.mean(np.abs(reconstructed_mean - original_mean))

    print(f"{name} MSE: {mse:.6f}, MAE: {mae:.6f}")
    return reconstructed_mean, mse, mae

In [ ]:
def build_uncalibrated(s):
    s.build_G(problem_type="bremsstrahlung")


def build_calibrated_raw(s):
    s.build_G(problem_type="bremsstrahlung", use_calibration=True, use_poly=False)


def build_calibrated_poly(s, d):
    s.build_G(
        problem_type="bremsstrahlung",
        use_calibration=True,
        use_poly=True,
        degree=d,
        weighted=True,
    )

In [ ]:
rec_uncal, mse_uncal, mae_uncal = run_decomp("uncalibrated", build_uncalibrated)
rec_raw, mse_raw, mae_raw = run_decomp("peak fit", build_calibrated_raw)

In [ ]:
MAX_DEGREE = 3

rec_poly, mse_poly, mae_poly = [], [], []

for d in range(MAX_DEGREE + 1):
    rec, mse, mae = run_decomp(
        f"poly fit deg {d}", lambda s, d=d: build_calibrated_poly(s, d)
    )
    rec_poly.append(rec)
    mse_poly.append(mse)
    mae_poly.append(mae)

In [ ]:
energy_axis = s.axes_manager.signal_axes[0].axis

plt.figure()
plt.plot(
    energy_axis, original_mean, label="original", color="k", alpha=0.8, linewidth=2
)
plt.plot(
    energy_axis, rec_uncal, label=f"uncalibrated (MSE: {mse_uncal:.6f})", linestyle="--"
)
plt.plot(energy_axis, rec_raw, label=f"peak fit (MSE: {mse_raw:.6f})", linestyle=":")

for d in range(MAX_DEGREE + 1):
    plt.plot(
        energy_axis,
        rec_poly[d],
        label=f"poly deg {d} (MSE: {mse_poly[d]:.6f})",
        linestyle="-.",
    )

plt.xlabel("Energy (keV)")
plt.ylabel("Intensity (counts)")
# plt.title("")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()